# 第 09 课 · 随机数其实是排队领号（浏览器版）

配套课文：[第 09 课 · 随机数其实是排队领号（网页版）](../docs/m0-foundations/09-making-random-variables.md)。

**怎么用**：本地装了 Jupyter 就直接打开这个文件；不想装任何东西，打开 [https://jupyter.org/try](https://jupyter.org/try)（或 Colab），把这一页整份放进去，每一格按 `Shift + Enter` 从上往下跑。

只用 Python 标准库（`random` 和 `math`），不需要安装任何东西。

**每格跑出来的数字和课文里印的逐字一致**（种子钉在 7）。

## 一、你已经会抽签，但只会抽一种

第 04 课你用过 `random.random()`，它给你一个 0 到 1 之间的数；你还用过 `random.uniform(25, 55)` 造每天的花销。

到这里你可能一直没好意思问：**如果我想造的东西不是"25 到 55 之间等可能"呢？**

比如外卖 App 说三十分钟到。真实情况是：有五成的机会正好三十分钟左右，两成很早就到，三成拖到一个小时。这个形状不均等，你手上那个"区间里随挑"根本造不出来。

再比如你手上只有过去七条内容的播放量记录，没有任何公式，可你想知道"下个月三十条大概能带来多少总播放"。

这两种需求其实是同一件事：**从"等可能的抽签"变成"按我给的形状抽签"。** 这节课就解决这个，而且解决方式简单到有点不像话。

## 二、那条一米长的队伍

先把 `random.random()` 到底给了你什么想清楚。

想象一条一米的队伍，你站在最前面那个位置，电脑往后随便指一个位置，指到哪儿你就站到哪儿。**每个位置被指到的机会一样多。** 位置用数字表示，就是 0 到 1 之间那个小数——0 是队头，1 是队尾。

这就是均匀分布的全部内容，也是所有随机的起点（"均匀"就是指位置等可能）。原书 §2.1 整节在讲"电脑怎么把这条队伍造出来"，答案是它造的是伪随机数、有周期，但周期长到你用不完，所以够用。这一节我们只管**用**这条队伍。

关键的一步在这里：**你不需要每个形状都另找一台抽签机器。队伍就一条，你可以拿它当任何东西。**

因为队伍上的一段，长度就代表机会。

## 三、离散：把队伍切成段

送达时间你想定成三档：两成早、五成准、三成晚。

做法是把一米长的队伍切成三段，长度分别是 0.2、0.5、0.3。抽签落在哪一段，就出哪一档。

先把课文那个文件开头的两行导入放好，再定义查队伍的那台机器（这两格都只是定义，运行它们不会打印任何东西）：

In [1]:
import math
import random

In [2]:
def pick(u, table):
    """u 是 0 到 1 之间的抽签；table 是 [(上沿, 名字)]，从小到大排好。"""
    for edge, name in table:
        if u < edge:
            return name
    return table[-1][1]

table = [(0.2, "快（20分钟）"), (0.7, "准点（30分钟）"), (1.0, "晚（50分钟）")]

注意那个 `0.2、0.7、1.0`：**它们不是三档的概率，是三段的"右端点"**，也就是累积起来的位置。两成 → 第一段到 0.2；五成是累积到 0.7；剩下的到 1.0。这就是这一节课唯一的技术点，其余全是它的变形。

跑两万次数一数（`random.seed(7)`）：

In [3]:
random.seed(7)
table = [(0.2, "快（20分钟）"), (0.7, "准点（30分钟）"), (1.0, "晚（50分钟）")]
counts = {"快（20分钟）": 0, "准点（30分钟）": 0, "晚（50分钟）": 0}
for _ in range(20000):
    counts[pick(random.random(), table)] += 1
for name, c in counts.items():
    print(f"{name}：{c / 20000 * 100:.1f}%")

快（20分钟）：20.0%
准点（30分钟）：50.2%
晚（50分钟）：29.8%


你要的是 20%、50%、30%，机器给你 20.0、50.2、29.8——差的都是第 07、08 课说的那种抽签毛边。

**这件事的名字叫逆变换**，正式写法是"拿一个均匀抽签 u，去查那个累积到 u 的位置对应的结果"。名字先放着，你刚才已经做完了：切成段，看落在哪段。以后你在原书 §2.2 看到的公式，就是这个动作换了身衣服。

## 四、连续：区间里随挑，和"多久来一单"

离散是切段，连续是**拉直**：把 0 到 1 这段队伍整段拉长到 25 到 55（下面这一格只是定义，运行它不会打印任何东西）：

In [4]:
def uniform_between(u, low, high):
    return low + (high - low) * u

队伍上 0.3 的位置，拉到 25 + 30 × 0.3 = 34。两万次跑出来平均 **39.93**，理论值正好是区间中点 40。

In [5]:
random.seed(7)
values = [uniform_between(random.random(), 25, 55) for _ in range(20000)]
print(f"平均 {sum(values) / len(values):.2f}（理论 {(25 + 55) / 2}）")

平均 39.93（理论 40.0）


稍微难点的形状：假设你店里平均每小时来 6 单，你想知道"两单之间的间隔"长什么样。这一类"等到某件事发生要多久"的形状有个名字叫指数分布，造法是一行（同样是定义，不打印东西）：

In [6]:
def exponential(u, rate):
    return -math.log(1 - u) / rate

`math.log` 是自然对数，`rate` 是"平均每小时几单"。为什么是这一行，进阶框里给严格的一小步；这里先直接看它对不对。跑两万次，`rate = 6`：

In [7]:
random.seed(7)
gaps = [exponential(random.random(), 6) for _ in range(20000)]
print(f"平均间隔 {sum(gaps) / len(gaps) * 60:.1f} 分钟（理论 10 分钟）")
short = 0
for g in gaps:
    if g < 1 / 12:
        short += 1
print(f"不到 5 分钟就来一单的比例：{short / len(gaps) * 100:.1f}%")

平均间隔 10.0 分钟（理论 10 分钟）
不到 5 分钟就来一单的比例：40.0%


一小时 6 单，平均十分钟来一单——机器给 10.0，对上了。

**现在看第二行，它是这一节课第一个真正反直觉的事实。** 平均间隔十分钟，可"不到五分钟就来一单"的情况占了四成。四成！而"间隔正好落在平均附近"远没有那么常见。

这不是机器出错。到达类的事情天生这样：你等的时间分布是**一边长尾巴**的——短间隔非常多，偶尔一次特别长把平均拽到十分钟。

这条规律生活里到处能用，而几乎没有凭直觉的人会答对：

**等一件"平均十分钟"的事，你有六成的机会比十分钟还早（两万遍里是 63.3%），也有少数几次会被拖得很久。** 所以"预计 30 分钟送达"这句话，如果你按 30 分钟来安排，你会经常提前空等、偶尔严重迟到——因为它说的是平均值，不是你会遇到的那一次。第 05 课讲的"平均不是典型"在这里以另一种形式回来了：**同一个平均，可以配完全不同的形状。**

把第三节那三档和第四节这条曲线画在同一张图上，"拿一条队伍当任何东西"就看得见了。左格是映射本身：横轴是队伍上的位置，纵轴是"抽到这儿该等多久"，橙台阶是三档送达，蓝线是每小时 6 单反解出来的曲线，蓝点是两万遍里挑出来的几百个真实抽签。右格是把蓝线抽两万遍落出来的形状——橙色那一块就是"不到 5 分钟"，占了四成，而平均（绿虚线）在十分钟后头。

![左格：横轴是队伍位置 0 到 1，纵轴是等多少分钟；橙色台阶在 0.2、0.7 处跳到 20、30、50 分钟，蓝线从左下角扬起、在 u=0.99 处冲到 50 分钟；队伍前 39.3% 那一段标出"全落在 5 分钟以内"。右格：两万遍抽出的间隔按分钟分堆，第一格最高，尾巴一路拖到 45 分钟以后；不到 5 分钟占 40.0%，平均 10.0 分钟](https://raw.githubusercontent.com/JavenZheng/monte-carlo-finance/main/assets/fig09_inverse_transform.svg)

图由 [code/fig09_inverse_transform.py](../code/fig09_inverse_transform.py) 生成，就是上面这一轮抽签（种子 7、两万次）。

顺带，`1 - u` 那个写法你可能会问：直接写 `u` 不行吗？行。队伍从哪头开始数都一样，因为 u 和 1−u 是同一个形状（进阶框有一句解释）。用 `1-u` 只是为了避开 `log(0)` 那个边界意外。

## 五、最难也最值钱的那种：你只有历史，没有公式

上面那些形状是你自己定的概率。可现实里你更常见的情形是：**你不知道任何概率，你只有一堆过去发生过的事。**

比如你的七条播放量：`820、1200、3400、610、950、2100、480`。你想知道下个月三十条大概总共能跑到多少。

办法朴素得让人怀疑：把这七条本身当成那条一米长的队伍——七条各占七分之一（每条的机会相等，因为你没有理由说哪条更该重现），然后把"从这七条里有放回抽三十次"重复一万遍，数出来的就是答案（下面这一格只是定义，运行它不会打印任何东西）：

In [8]:
def future_total(records, k, runs):
    """从历史 records 里有放回抽 k 条，看这 k 条加起来是多少；重复 runs 次。"""
    out = []
    for _ in range(runs):
        drawn = random.choices(records, k=k)
        out.append(sum(drawn))
    out.sort()
    return out

`random.choices(列表, k=30)` 就是"有放回抽三十次"，第 04 课你已经用过它。跑一万遍"未来的三十条"（种子 7）：

In [9]:
plays = [820, 1200, 3400, 610, 950, 2100, 480]
random.seed(7)
totals = future_total(plays, 30, 10000)
p10 = totals[int(0.1 * len(totals))]
p50 = totals[int(0.5 * len(totals))]
p90 = totals[int(0.9 * len(totals))]
print(f"三十条的总播放：一成低于 {p10:.0f}，中位 {p50:.0f}，九成不超过 {p90:.0f}")
print(f"最常在哪个区间：{totals[int(0.25*len(totals))]:.0f} 到 {totals[int(0.75*len(totals))]:.0f}")

三十条的总播放：一成低于 34190，中位 40830，九成不超过 47910
最常在哪个区间：37230 到 44490


**这就是一个可以直接说出口的结论**："按我这七条的形状，下个月三十条的总播放大概四万，保守说三万四以上，乐观不超过四万八。"

而且它顺手解决了一件事：你完全没有假定播放量服从任何有名有姓的分布。这个做法有个正式名字叫**经验分布**——把你手上那一列记录当成可能性表；重复抽它、看统计量怎么变，叫 **bootstrap**。名字留在备忘卡，你今天拿到的是动作。

顺手验一下它的合理性：七条的平均是 1365.7，乘三十 = 40971，而模拟的中位是 40830。第 06 课那句话在这里正好闭环：**期望（40971）是你想要的那个数，模拟（40830）是你拿到它的手段。**

## 六、今晚就能用起来，但三件事要先说清

代码在 [code/lesson09_making_any_distribution.py](../code/lesson09_making_any_distribution.py)。你要改的只有第五节那一列 `plays`——换成你自己的任何记录：这半年的月收入、这一年的账单、所有跑过的步数。

三种现实用法，都是同一段代码：

**"下个月总播放大概多少"**（把三十换成你实际的发文条数）。

**"我这个月打车费会到多少"**：把历史每一笔金额粘进去，一次抽二十二天，取总和。这比第 04 课那个"每天 25 到 55"更实在，因为那个区间是我编的，而这一列金额是你真花过的。

**"客服要排几个人"**：如果你有历史每天的问题量，同样的动作给你"九成不超过多少"，那就是排班要看的数（第 05 课练习三的答案口径：**预算和排班按分位数定，不按平均定**）。

三件事必须说清，因为这套办法的毛病全部藏在这里：

**第一，你抽不出你历史上没出现过的形状。** 那七条里最大 3400，所以一万遍里永远不会出现一条五万的。你的"下个月最好情况"被你的历史封顶了。如果你的账号其实有可能爆一条大的——而它正是你经营它的理由——这套办法会系统性低估你的上限。这就是第 05 课那条尾巴、原书 §9.3（书页 506）整节在处理的对象：**样本给不了你没见过的事。**

**第二，有放回地重抽历史，等于假设"下个月和过去这七条是同一个世界"。** 你换了选题、换了平台推荐策略、或者签了 MCN，这个假设当场断掉。断掉不是错误，是你必须知道它断了——就像第 01 课那个杯子，你把概率改了，答案才跟着改。

**第三，条数太少时，第五节那三个数给的是一种虚假的精确。** 七条记录抽三十条，"一成低于 34190"这个数本身是用一万次抽签量的，它自己也有毛边（第 07 课那套算式适用于它）。七条数据报出"34190"这种四位数精度，是在装饰自己。取整说"三万四到四万八"就够了。

## 这一课带走的一句话

一台均匀抽签就够：把队伍切成段就是离散，整段拉直就是均匀，反过来解就是指数，把历史当成队伍就是没有公式时的答案——但队伍以外的事，它永远抽不出来。

配套课文：[第 09 课 · 随机数其实是排队领号（网页版）](../docs/m0-foundations/09-making-random-variables.md)。

---

**许可**：正文 CC BY-NC-SA 4.0，代码 MIT，见 [LICENSE.md](../LICENSE.md)。

**建议购买正版**：Paul Glasserman, *Monte Carlo Methods in Financial Engineering*, Springer, 2003；中译本《金融工程中的蒙特卡罗方法》。本 notebook 未复制原书任何文字、图表或表格。
